<a href="https://colab.research.google.com/github/JulTob/Modelling/blob/main/GLM/GLM_2_region_agencia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

equipo	N	m_sin	disp_sin	p0_fraude	p_evento	p_baja	m_coste
equipo_07	260107	1.1	1.1	1	1.1	0.7	1.7	1.05	0.56


In [ ]:
library(tidyverse)

# Fuente de datos del equipo 07, fijada para reproducibilidad
repositorio <- "https://raw.githubusercontent.com/JulTob/Modelling/698dad06c4228ea79319fdba27647a0c61474703/GLM/"
cartera <- read_csv(paste0(repositorio, "cartera.csv"), show_col_types = FALSE)

# Una fila representa una póliza; n_gestiones es el conteo observado.
stopifnot(
  nrow(cartera) > 0,
  all(c("id_poliza", "id_region", "id_agencia", "n_gestiones") %in% names(cartera))
)
glimpse(cartera)
summary(cartera$n_gestiones)

## 🔰 Modelo anidado: región → agencia → póliza

**Pregunta:** ¿cómo se distribuye el número de gestiones entre las pólizas, las agencias y las regiones? ¿Qué agencias destacan por encima o por debajo de su región?

Cada **fila** de `cartera` representa una póliza. Distinguimos tres magnitudes:

- `n_gestiones` = **Y**: número de gestiones de una póliza (variable respuesta).
- **N** = número de pólizas que pertenecen a la agencia o región (tamaño del grupo).
- **G** = suma de gestiones de todas las pólizas del grupo.

La media de gestiones por póliza es **G / N**. Una agencia grande puede tener más gestiones totales sin que cada póliza requiera más gestiones.

**Estructura:** región contiene agencias; agencia contiene pólizas. Creamos un identificador combinado `agencia_anidada` para evitar que agencias con el mismo código en regiones diferentes se confundan.

**Lectura de las figuras:** mantenemos el estilo del Caso 1, con un **violín y una vela (caja + bigotes) interior**. Como `n_gestiones` es discreta, el violín suaviza una distribución de enteros: sus valores intermedios no son datos observados. Añadimos después un **gráfico de frecuencias** que responde exactamente a «¿cuántas pólizas registran k gestiones?».

En este cuaderno el objetivo es **gestiones por póliza**, no gestiones por unidad de tiempo: por ahora no introducimos `exposicion` como offset. Si las pólizas tienen períodos de observación distintos y queremos una **intensidad temporal**, esa será otra pregunta estadística.

In [ ]:
# 1. Verificar y construir los tres niveles observacionales
stopifnot(
  all(c("id_poliza", "id_region", "id_agencia", "n_gestiones") %in% names(cartera))
)

datos_gestiones <- cartera %>%
  filter(!is.na(id_poliza), !is.na(id_region),
         !is.na(id_agencia), !is.na(n_gestiones)) %>%
  mutate(
    region = factor(id_region),
    agencia = factor(id_agencia),
    agencia_anidada = interaction(region, agencia, sep = ":", drop = TRUE)
  )

stopifnot(
  nrow(datos_gestiones) > 0,
  anyDuplicated(datos_gestiones$id_poliza) == 0,
  all(is.finite(datos_gestiones$n_gestiones)),
  all(datos_gestiones$n_gestiones >= 0),
  all(datos_gestiones$n_gestiones == floor(datos_gestiones$n_gestiones))
)

# Resumir las pólizas de cada agencia (nivel 2)
resumen_agencias <- datos_gestiones %>%
  group_by(region, agencia, agencia_anidada) %>%
  summarise(
    n_polizas = n(),
    gestiones_totales = sum(n_gestiones),
    gestiones_por_poliza = mean(n_gestiones),
    .groups = "drop"
  )

# Resumir las pólizas de cada región (nivel 1)
resumen_regiones <- datos_gestiones %>%
  group_by(region) %>%
  summarise(
    n_agencias = n_distinct(agencia_anidada),
    n_polizas = n(),
    gestiones_totales = sum(n_gestiones),
    gestiones_por_poliza = mean(n_gestiones),
    .groups = "drop"
  )

# Incorporar N al eje horizontal de los gráficos sin repetir filas
datos_gestiones <- datos_gestiones %>%
  left_join(
    select(resumen_agencias, region, agencia_anidada, n_polizas),
    by = c("region", "agencia_anidada")
  ) %>%
  mutate(
    agencia_etiqueta = factor(
      paste0("Ag. ", as.character(agencia), "\nN=", n_polizas)
    )
  )

etiquetas_regiones <- setNames(
  paste0("Región ", resumen_regiones$region,
         "\nN=", resumen_regiones$n_polizas),
  as.character(resumen_regiones$region)
)

paleta_regiones <- setNames(
  grDevices::hcl.colors(nlevels(datos_gestiones$region), palette = "Dark 3"),
  levels(datos_gestiones$region)
)

cat("Pólizas:", nrow(datos_gestiones),
    "| Regiones:", nrow(resumen_regiones),
    "| Agencias anidadas:", nrow(resumen_agencias), "\n")
cat("Pólizas con datos ausentes omitidas:",
    nrow(cartera) - nrow(datos_gestiones), "\n")

print(resumen_regiones, n = Inf)
print(resumen_agencias, n = Inf)

In [ ]:
# 2. Violín + vela por región
grafico_regiones <- ggplot(
  datos_gestiones,
  aes(x = region, y = n_gestiones, fill = region)
) +
  geom_violin(
    trim = TRUE, scale = "width",
    alpha = 0.35, linewidth = 1, colour = "grey25"
  ) +
  geom_boxplot(
    width = 0.18, fill = "white", colour = "grey15",
    alpha = 0.95, linewidth = 0.8, outlier.shape = NA
  ) +
  scale_fill_manual(values = paleta_regiones) +
  scale_x_discrete(labels = etiquetas_regiones) +
  labs(
    title = "Gestiones por póliza, según región",
    subtitle = "Violín + caja interior; N indica el número de pólizas",
    x = "Región y pólizas observadas", y = "Número de gestiones por póliza"
  ) +
  guides(fill = "none") +
  theme_minimal(base_size = 12)

print(grafico_regiones)

In [ ]:
# 3. Violín + vela por agencia, anidada y coloreada por región
# El identificador agencia_anidada distingue agencias de regiones distintas.
# Los grupos con menos de 3 pólizas se muestran solo mediante la caja.
grafico_agencias <- ggplot(
  datos_gestiones,
  aes(x = agencia_etiqueta, y = n_gestiones,
      fill = region, group = agencia_anidada)
) +
  geom_violin(
    data = filter(datos_gestiones, n_polizas >= 3),
    trim = TRUE, scale = "width",
    alpha = 0.35, linewidth = 0.8, colour = "grey25"
  ) +
  geom_boxplot(
    width = 0.18, fill = "white", colour = "grey15",
    alpha = 0.95, linewidth = 0.7, outlier.shape = NA
  ) +
  facet_wrap(~ region, scales = "free_x", ncol = 2) +
  scale_fill_manual(values = paleta_regiones) +
  labs(
    title = "Gestiones por póliza, según agencia",
    subtitle = "Cada agencia está dentro de una región; N = pólizas",
    x = "Agencia y número de pólizas", y = "Número de gestiones por póliza",
    fill = "Región"
  ) +
  theme_minimal(base_size = 12) +
  theme(
    axis.text.x = element_text(angle = 65, hjust = 1),
    legend.position = "bottom"
  )

print(grafico_agencias)

In [ ]:
# 4. Complemento: cuántas pólizas registran cada número de gestiones
# Esta es la representación discreta exacta; no suaviza los conteos.
frecuencias_gestiones <- datos_gestiones %>%
  count(region, n_gestiones, name = "n_polizas")

grafico_frecuencias <- ggplot(
  frecuencias_gestiones,
  aes(x = n_gestiones, y = n_polizas, fill = region)
) +
  geom_col(width = 0.85) +
  facet_wrap(~ region, scales = "free_y", ncol = 2) +
  scale_fill_manual(values = paleta_regiones) +
  labs(
    title = "¿Cuántas pólizas tienen k gestiones?",
    subtitle = "Conteo de pólizas para cada valor entero de n_gestiones",
    x = "Número de gestiones por póliza", y = "Cantidad de pólizas"
  ) +
  guides(fill = "none") +
  theme_minimal(base_size = 12)

print(grafico_frecuencias)

## ⚙️ Del diagrama al modelo anidado

Para una póliza \(i\), en la agencia \(a\) de la región \(r\):

\[
Y_{[r,a,i]} \sim \mathrm{Poisson}(\mu_{[r,a,i]})
\]

\[
\log \mu_{[r,a,i]} = \beta_0 + u_r + v_{[r,a]}
\]

- \(\beta_0\): nivel general de gestiones, en escala logarítmica.
- \(u_r\): desviación particular de una región.
- \(v_{[r,a]}\): desviación de una agencia **respecto a su región**.

Suponemos \(u_r\sim N(0,\sigma^2_{\mathrm{region}})\) y \(v_{[r,a]}\sim N(0,\sigma^2_{\mathrm{agencia}})\), independientes condicionalmente. Es un modelo de **interceptos aleatorios**; permite estimar qué proporción de la heterogeneidad corresponde a regiones y agencias.

En R, `(1 | region/agencia)` representa dos componentes: `(1 | region)` y `(1 | region:agencia)`. Usaremos el identificador `agencia_anidada` para que la fórmula resulte más legible.

⚠️ La póliza es aquí **una observación**, no un tercer efecto aleatorio: con una sola fila por póliza no podemos separar de forma ordinaria un intercepto propio de cada póliza de la variación residual sin hipótesis adicionales.

**¿Dónde entra N, la cantidad de pólizas?** Si en vez de observar cada póliza trabajásemos con el total de gestiones de cada agencia y supusiéramos la misma media condicional por póliza:

\[
G_{[r,a]} \sim \mathrm{Poisson}(N_{[r,a]} \mu_{[r,a]})
\]

y por tanto \(\log E[G_{[r,a]}]=\log N_{[r,a]}+\log \mu_{[r,a]}\). **Solo al agregar** aparece el `offset(log(n_polizas))`. En el modelo que conserva **una fila por póliza**, no hace falta incorporar N como offset.

**Diagnóstico pendiente:** un conteo puede presentar sobredispersión o ceros excesivos; Poisson es el punto de partida, no una conclusión automática.

In [ ]:
# 5. Ajuste opcional: modelo de Poisson con región y agencia anidada
# La ejecución puede tardar con una cartera grande.
if (!requireNamespace("lme4", quietly = TRUE)) {
  message("Falta lme4. Instálalo con: install.packages('lme4')")
} else if (nlevels(datos_gestiones$region) < 2 ||
           nlevels(datos_gestiones$agencia_anidada) <= nlevels(datos_gestiones$region)) {
  message("Se necesitan varias regiones y más agencias que regiones para el modelo anidado.")
} else {
  modelo_jerarquico <- lme4::glmer(
    n_gestiones ~ 1 + (1 | region) + (1 | agencia_anidada),
    data = datos_gestiones,
    family = poisson(link = "log"),
    control = lme4::glmerControl(optimizer = "bobyqa")
  )

  print(summary(modelo_jerarquico))
  print(lme4::VarCorr(modelo_jerarquico))

  # Los efectos se expresan como multiplicadores respecto al nivel general
  efectos <- lme4::ranef(modelo_jerarquico)
  efectos_regiones <- tibble(
    region = rownames(efectos$region),
    efecto_log = efectos$region[["(Intercept)"]],
    multiplicador = exp(efecto_log)
  )
  efectos_agencias <- tibble(
    agencia_anidada = rownames(efectos$agencia_anidada),
    efecto_log = efectos$agencia_anidada[["(Intercept)"]],
    multiplicador = exp(efecto_log)
  )

  print(efectos_regiones, n = Inf)
  print(efectos_agencias, n = Inf)

  dispersion_pearson <- sum(residuals(modelo_jerarquico, type = "pearson")^2) /
    df.residual(modelo_jerarquico)
  cat("Dispersión de Pearson aproximada:", round(dispersion_pearson, 2), "\n")
}

### ⚜️ Qué contrastaremos después de ejecutar el cuaderno

1. **Cantidad de pólizas:** comparar \(N\), las pólizas por región y agencia, antes de interpretar gestiones totales.
2. **Distribución del número de gestiones:** comparar las velas y los violines; el gráfico de barras permite comprobar los conteos enteros y el peso de los ceros.
3. **Modelo jerárquico:** interpretar \(\exp(u_r)\) y \(\exp(v_{[r,a]})\) como multiplicadores condicionales de la media de gestiones por póliza.
4. **Validez del modelo:** examinar dispersión, ceros, convergencia y posibles covariables de las pólizas. Un efecto aparente de agencia podría deberse a diferencias entre las pólizas que administra.

Este cuaderno usa **gestiones por póliza** como objeto principal. Una futura tasa por tiempo de exposición requerirá justificar esa unidad de análisis por separado.